# Import Library

In [1]:
import os
import pandas as pd
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import f1_score,classification_report, silhouette_score, roc_auc_score, ConfusionMatrixDisplay

# Data Loading

In [2]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv')['Risk_Level']
y_test = pd.read_csv('../data/processed/y_test.csv')['Risk_Level']

X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19289 entries, 0 to 19288
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Financial_Stability_Score        19289 non-null  float64
 1   On_Time_Delivery_Rate            19289 non-null  float64
 2   Defect_Rate                      19289 non-null  float64
 3   Lead_Time_Days                   19289 non-null  float64
 4   Alternative_Suppliers_Available  19289 non-null  float64
 5   Contract_Length_Months           19289 non-null  float64
 6   Environmental_Compliance         19289 non-null  float64
 7   Previous_Disruptions             19289 non-null  float64
dtypes: float64(8)
memory usage: 1.2 MB


# Train Model

In [4]:
def make_pipeline(model):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', model),
    ])

models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=2000),
    'RandomForestClassifier': RandomForestClassifier(class_weight='balanced', n_estimators=300, min_samples_leaf=5, n_jobs=1, random_state=42),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, model in models.items():
    pipe = make_pipeline(model)
    f1 = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    results[name] = {'F1': f1.mean(), 'ROC-AUC': auc.mean()}
pd.DataFrame(results).T.sort_values('ROC-AUC', ascending=False)


,F1,ROC-AUC
HistGradientBoosting,0.888929,0.933252
RandomForestClassifier,0.899655,0.930065
LogisticRegression,0.877135,0.921934
